# EEG-Bench Results Analysis

This notebook analyzes benchmark results from EEG-Bench experiments.

**Features:**
- Load all experiment results from JSON files
- Data efficiency curves (accuracy vs. training data %)
- Model comparison tables and heatmaps
- BCI vs Clinical task analysis

## 1. Setup & Data Loading

In [1]:
import json
import glob
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, cohen_kappa_score
from pathlib import Path

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Task categorization
BCI_TASKS = ['Five Fingers MI', 'Left Hand vs Right Hand vs Feet vs Tongue MI',]
CLINICAL_TASKS = [
    "parkinsons", "schizophrenia", "mtbi", "ocd", "epilepsy",
    "abnormal", "sleep_stages", "seizure", "binary_artifact", "multiclass_artifact"
]

# Friendly names for display
TASK_DISPLAY_NAMES = {
    "left_right": "Left/Right Hand",
    "right_feet": "Right Hand/Feet",
    "left_right_feet_tongue": "4-Class MI",
    "5_fingers": "5 Fingers",
    "parkinsons": "Parkinson's",
    "schizophrenia": "Schizophrenia",
    "mtbi": "mTBI",
    "ocd": "OCD",
    "epilepsy": "Epilepsy",
    "abnormal": "Abnormal EEG",
    "sleep_stages": "Sleep Stages",
    "seizure": "Seizure",
    "binary_artifact": "Artifact (Binary)",
    "multiclass_artifact": "Artifact (Multi)"
}

MODEL_DISPLAY_NAMES = {
    "LaBraMModel": "LaBraM",
    "BENDRModel": "BENDR",
    "NeuroGPTModel": "NeuroGPT",
    "REVEBenchmarkModel": "REVE",
    "REVEClinicalModel": "REVE",
    "LeJEPABCI": "LeJEPA",
    "LeJEPAClinical": "LeJEPA",
    "CSPLDAModel": "CSP-LDA",
    "CSPSVMModel": "CSP-SVM",
    "BrainfeaturesLDAModel": "LDA",
    "BrainfeaturesSVMModel": "SVM",
    "LUNAModel": "LUNA"
}

In [2]:

from collections.abc import Sequence

from tqdm import tqdm


def _flatten_label_structure(values):
    """Recursively flatten nested predictions/labels regardless of depth."""
    if values is None:
        return []
    if isinstance(values, np.ndarray):
        return values.reshape(-1).tolist()
    if isinstance(values, Sequence) and not isinstance(values, (str, bytes)):
        flattened = []
        for item in values:
            flattened.extend(_flatten_label_structure(item))
        return flattened
    return [values]


def compute_metrics(y_true, y_pred):
    """Compute classification metrics from predictions (robust to nesting)."""
    y_true_flat = _flatten_label_structure(y_true)
    y_pred_flat = _flatten_label_structure(y_pred)

    if not y_true_flat or not y_pred_flat:
        return {
            "accuracy": np.nan,
            "balanced_accuracy": np.nan,
            "f1_weighted": np.nan,
            "f1_macro": np.nan,
            "kappa": np.nan,
        }

    min_len = min(len(y_true_flat), len(y_pred_flat))
    y_true_arr = np.asarray(y_true_flat[:min_len])
    y_pred_arr = np.asarray(y_pred_flat[:min_len])

    metrics = {
        "accuracy": accuracy_score(y_true_arr, y_pred_arr),
        "balanced_accuracy": balanced_accuracy_score(y_true_arr, y_pred_arr),
        "f1_weighted": f1_score(y_true_arr, y_pred_arr, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_true_arr, y_pred_arr, average="macro", zero_division=0),
        "kappa": cohen_kappa_score(y_true_arr, y_pred_arr),
    }
    return metrics


def _parse_eval_noise(data: dict) -> dict:
    """Parse evaluation noise metadata with backward compatibility."""
    eval_noise = data.get("eval_noise") or {}
    tag = eval_noise.get("tag", "clean")
    snr_db = eval_noise.get("snr_db")
    noise_types = eval_noise.get("noise_types") or []
    is_clean = snr_db is None
    return {
        "noise_tag": tag,
        "noise_snr_db": snr_db,
        "noise_types": noise_types,
        "is_clean": is_clean,
    }


def load_results(results_dir="results/raw"):
    """Load all results JSON files into a DataFrame."""
    records = []

    json_files = glob.glob(os.path.join(results_dir, "*.json"))
    print(f"Found {len(json_files)} result files")

    for filepath in tqdm(json_files):
        try:
            with open(filepath) as f:
                data = json.load(f)

            task_name = data.get("task_name", "unknown")
            model_names = data.get("models_names", ["unknown"])
            percentage = data.get("data_percentage", 1.0)
            linear_probe = data.get("linear_probe", False)
            timestamp = data.get("timestamp", "")
            data_stats = data.get("data_stats", {})
            ckpt_id = data.get("checkpoint_id", {})
            step = 'None'
            result_prefix = data.get("result_prefix", "").replace('lejepa', '')
            if ckpt_id:
                step = ckpt_id #int(ckpt_id.split('step_')[-1]) if 'last' not in ckpt_id else 

            noise_info = _parse_eval_noise(data)
            probe_type = data.get("probe_type", "linear") #linear or attn

            y_test = data.get("y_test", [[]])
            results = data.get("results", [[]])

            # Process each model's results
            for i, model_name in enumerate(model_names):
                try:
                    # Get predictions for this model (handle nested structure)
                    if i < len(results):
                        y_pred = results[i] if isinstance(results[i], list) else results
                        y_true = y_test[i] if i < len(y_test) and isinstance(y_test[i], list) else y_test
                    else:
                        y_pred = results[0] if results else []
                        y_true = y_test[0] if y_test else []

                    # Compute metrics
                    metrics = compute_metrics(y_true, y_pred)

                    # Clean up model name for display
                    display_name = MODEL_DISPLAY_NAMES.get(model_name, model_name)

                    record = {
                        "task": task_name,
                        "task_display": TASK_DISPLAY_NAMES.get(task_name, task_name),
                        "task_type": "BCI" if task_name in BCI_TASKS else "Clinical",
                        "model": model_name + result_prefix,
                        "model_display": display_name + result_prefix,
                        "percentage": percentage,
                        "linear_probe": linear_probe,
                        "timestamp": timestamp,
                        # "total_samples": data_stats.get("total_samples", 0),
                        "file": os.path.basename(filepath),
                        "step": step,
                        "probe_type": probe_type,
                        **noise_info,
                        **metrics,
                    }
                    records.append(record)

                except Exception:
                    raise

        except Exception:
            raise

    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} experiment results")
    return df


In [ ]:
df_all = load_results("results/raw")


Found 926 result files


  0%|          | 3/926 [00:00<00:57, 15.96it/s]

In [ ]:
include_baselines = True

In [ ]:
df_all['model_display'] = df_all['model_display'].str.replace('lejeba', '')
df_all['model_display'].unique()

In [ ]:
prefix_models = ['LeJEPA_base_global_projv2_02sig_FULL', 
                'LeJEPA_base_global_projv2_smallds',
                'LeJEPA_base_global_projv2_02sig_FULL_v2',
                'LeJEPA_base_global_projv2_05sig_FULL_v2',
                'LeJEPA_vitb_global_proj_05sig_FULL_v2',
                'LeJEPA_base_global_projv2_05sig_FULL_v2_longer'
                ]# 'LeJEPA_base_global_projv2_smallds_20k', 'LeJEPA_base_v2_noproj']
df_all = df_all[df_all['model_display'].str.startswith(tuple(prefix_models))]

In [ ]:


renamer = {"LeJEPA_base_global_projv2_02sig_FULL":"Laya",
           "LeJEPA_base_global_projv2_smallds":"Laya-S", 
           "LeJEPA_base_global_projv2_02sig_FULL_v2":"Laya-V2", 
           "LeJEPA_base_global_projv2_05sig_FULL_v2":"Laya-V2-05sig",
           "LeJEPA_base_global_projv2_05sig_FULL_v2_longer":"Laya-V2-05sig_longer",
           "LeJEPA_vitb_global_proj_05sig_FULL_v2": "Laya-vitb-05sig",
           "LeJEPA_base_v2_noproj":"Laya-NoProj"}

import re
for old, new in renamer.items():
    df_all["model_display"] = df_all["model_display"].str.replace(f"^{re.escape(old)}", new, regex=True)


df_all.head()

In [ ]:
# df_all = df_all[df_all['model_display'].str.startswith(['Laya','Laya-S'])]

In [ ]:

# Load results (clean + noise-aware)
if include_baselines:
    df_labram = pd.read_csv('labram_results_all.csv', index_col = None)
    df_labram['noise_types'] = df_labram['noise_types'].apply(lambda x: eval(x))

    df_luna = pd.read_csv('luna_results.csv', index_col = 0)

    df_combined = pd.concat([df_all, df_labram, df_luna])
    df_combined['step'] = df_combined['step'].fillna('last')
    df_combined['is_clean'] = df_combined['is_clean'].fillna(True)
else:
    df_combined = df_all.copy()


df_combined['probe_type'] = df_combined['probe_type'].fillna('linear')


df_clean = df_combined.copy()
df_noise = df_combined.copy()

if len(df_combined) > 0:
    # Normalize noise types for grouping/labels
    if "noise_types" in df_combined.columns:
        df_combined["noise_types_str"] = df_combined["noise_types"].apply(
            lambda x: "+".join(x) if isinstance(x, (list, tuple)) else ""
        )
    else:
        df_combined["noise_types_str"] = ""

    df_clean = df_combined[df_combined["is_clean"]].copy() if "is_clean" in df_combined.columns else df_combined.copy()
    df_noise = df_combined[~df_combined["is_clean"]].copy() if "is_clean" in df_combined.columns else df_combined.iloc[0:0].copy()
    

    # IMPORTANT: all existing plots below should use clean-only results
    df = df_clean
    df = df[df['probe_type'] == 'linear']
    df = df[df['step'] == 'last']
    

    print(f"Clean results: {len(df_clean)} rows")
    print(f"Noise results: {len(df_noise)} rows")
    print(f"Unique tasks (clean): {df['task'].nunique()}")
    print(f"Unique models (clean): {df['model_display'].nunique()}")
    print(f"Data percentages (clean): {sorted(df['percentage'].unique())}")
    print(f"Tasks (clean): {df['task'].unique().tolist()}")
    print(f"Models (clean): {df['model_display'].unique().tolist()}")
    print(f"Steps (clean): {df['step'].unique().tolist()}")
else:
    print("No results found. Run experiments first with run_experiments.py")


In [ ]:
# Drop duplicates (exclude unhashable list columns like noise_types)
unhashable_cols = {"noise_types"}
exclude_cols = {"timestamp", "file"} | unhashable_cols

subset_cols = [c for c in df.columns if c not in exclude_cols]
df = df.drop_duplicates(subset=subset_cols)

print(f"After deduplication: {len(df)} rows")
print(f"Models (clean): {df['model_display'].unique().tolist()}")


In [ ]:
# Preview data
if len(df) > 0:
    display(df.head(10))

## 2. Data Efficiency Curves

Plots showing how model performance changes with varying amounts of training data.

In [ ]:
def plot_data_efficiency(df, task, metric="balanced_accuracy", figsize=(10, 6)):
    """Plot data efficiency curve for a single task."""
    task_df = df[df["task"] == task].copy()
    
    if len(task_df) == 0:
        print(f"No data for task: {task}")
        return
    
    plt.figure(figsize=figsize)
    
    models = task_df["model_display"].unique()
    colors = plt.cm.tab10(np.linspace(0, 1, len(models)))
    
    for model, color in zip(models, colors):
        model_df = task_df[task_df["model_display"] == model].sort_values("percentage")
        
        # Group by percentage and compute mean/std if multiple runs
        grouped = model_df.groupby("percentage")[metric].agg(["mean", "std"]).reset_index()
        
        plt.plot(grouped["percentage"] * 100, grouped["mean"],
                 marker='o', label=model, color=color, linewidth=2, markersize=8)
        
        # Add error bars if std available
        if grouped["std"].notna().any():
            plt.fill_between(grouped["percentage"] * 100,
                           grouped["mean"] - grouped["std"],
                           grouped["mean"] + grouped["std"],
                           alpha=0.2, color=color)
    
    task_display = TASK_DISPLAY_NAMES.get(task, task)
    plt.xlabel("Training Data (%)", fontsize=12)
    plt.ylabel(metric.replace("_", " ").title(), fontsize=12)
    plt.title(f"Data Efficiency: {task_display}", fontsize=14)
    plt.legend(loc="lower right", fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.xlim(0, 105)
    plt.ylim(0, 1.05)
    plt.tight_layout()
    plt.show()


def plot_all_efficiency_curves(df, metric="balanced_accuracy", tasks=None):
    """Plot data efficiency curves for all tasks."""
    if tasks is None:
        tasks = df["task"].unique()
    
    for task in tasks:
        plot_data_efficiency(df, task, metric)

In [ ]:
model_colors = {
    "Laya": "#1473d3",
    "Laya-S": "#3aacce",
    # "Laya-S_20k": "#4ece3a",

    'LaBraM': "#c51a1a"
}

In [ ]:
df_noise['model_display'].unique()

In [ ]:
def plot_efficiency_grid(df, metric="balanced_accuracy", ncols=3):
    """Plot all efficiency curves in a grid layout."""
    tasks = df["task"].unique()
    nrows = int(np.ceil(len(tasks) / ncols))
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
    axes = axes.flatten() if len(tasks) > 1 else [axes]
    
    models = df["model_display"].unique()
    colors = model_colors
    
    
    for idx, task in enumerate(tasks):
        ax = axes[idx]
        task_df = df[df["task"] == task]
        
        for model in models:
            if 'LUNA' in model: 
                continue
            if '20k' in model:
                continue
            model_df = task_df[task_df["model_display"] == model].sort_values("percentage")
            if len(model_df) > 0:
                grouped = model_df.groupby("percentage")[metric].mean().reset_index()
                ax.plot(grouped["percentage"] * 100, grouped[metric],
                       marker='o', label=model, color=colors[model], linewidth=2)
        
        ax.set_title(TASK_DISPLAY_NAMES.get(task, task), fontsize=11)
        ax.set_xlabel("Training Data (%)")
        ax.set_ylabel(metric.replace("_", " ").title())
        ax.set_xlim(0, 105)
        ax.set_ylim(0, 1.05)
        #logscale x
        # ax.set_xscale('log')
        ax.grid(True, alpha=0.3)
    
    # Hide empty subplots
    for idx in range(len(tasks), len(axes)):
        axes[idx].set_visible(False)
    
    # Add legend
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=min(5, len(models)),
              bbox_to_anchor=(0.5, 1.02), fontsize=10)
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    plt.savefig("results/efficiency_curves_grid.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved to results/efficiency_curves_grid.png")

In [ ]:
# # Plot grid view
# if len(df) > 0 and df["task"].nunique() > 1:
#     plot_efficiency_grid(df)

## 3. Model Comparison Tables

In [ ]:
def model_comparison_table(df, percentage=1.0, metric="balanced_accuracy"):
    """Create pivot table: tasks (rows) x models (columns)."""
    subset = df[df["percentage"] == percentage].copy()

    if len(subset) == 0:
        print(f"No data for percentage={percentage}")
        return None

    table = subset.pivot_table(
        values=metric,
        index="task_display",
        columns="model_display",
        aggfunc="mean"
    )

    # Add mean row
    table.loc["Mean"] = table.mean(axis=0)

    # Sort by mean performance (descending)
    table = table.loc[:, table.loc["Mean"].sort_values(ascending=False).index]

    os.makedirs("tables/", exist_ok=True)
    table.to_csv(f"tables/model_comparison_{metric}.csv")
    print(f"Saved model comparison table to tables/model_comparison_{metric}.csv")

    return table.round(3)


In [ ]:
TASK_DISPLAY = {
    "Five Fingers MI": "5-Finger MI",
    "Left Hand vs Right Hand MI": "LH vs RH MI",
    "Left Hand vs Right Hand vs Feet vs Tongue MI": "4-Class MI",
    "Right Hand vs Feet MI": "RH vs Feet MI",
    "abnormal": "Abnormal",
    "binary artifact": "Artifact (Binary)",
    "multiclass artifact": "Artifact (Multiclass)",
    "epilepsy": "Epilepsy",
    "mtbi": "mTBI",
    "ocd": "OCD",
    "parkinsons": "Parkinson's",
    "schizophrenia": "Schizophrenia",
    "seizure": "Seizure",
    "sleep stages": "Sleep Stages",
}



In [ ]:
# Model comparison tables for each metric (with LaTeX export)
metrics = ["accuracy", "balanced_accuracy", "f1_weighted", "f1_macro", "kappa"]

if len(df) > 0:
    for metric in metrics:
        table = model_comparison_table(df, percentage=1.0, metric=metric)
        # rename LUNAModel -> LUNA (if needed)
        table = table.rename(columns={"LUNAModel": "LUNA"})

        # enforce column order
        desired_order = ["LaBraM", "LUNA", "Laya-S", "Laya","Laya-V2-05sig", "Laya-V2-05sig_longer", "Laya-vitb-05sig"] # add more if needed
        cols = [c for c in desired_order if c in table.columns]# + [c for c in table.columns if c not in desired_order]
        table = table[cols]

        
        if table is None or len(table) == 0:
            continue
        print(f"Model Comparison (100% training data, {metric.replace('_', ' ').title()}):")
        display(table.style.background_gradient(cmap="RdYlGn", axis=None))
        

        caption = f"Linear probe performance on EEG-Bench ({metric})."
        label = f"tab:eegebench_{metric}"

        # clean table for LaTeX
        table = table.copy()
        table.index.name = None
        table.columns.name = None
        table.index = (
            table.index
            .str.replace("_clinical", "", regex=False)
            .str.replace("_", " ", regex=False)
        )
        # apply after your underscore cleanup
        table.index = table.index.to_series().map(lambda x: TASK_DISPLAY.get(x, x)).values


        latex_tab = table.to_latex(
            index=True,
            float_format="%.3f",
            escape=False,
            column_format="l" + "r" * len(table.columns),
            caption=caption,
            label=label,
        )

        latex_full = "\\begin{table}[t]\n\\centering\n"
        latex_full += latex_tab.replace("\\begin{table}", "").replace("\\end{table}", "")
        latex_full = latex_full.replace("\\begin{tabular}", "\\resizebox{\\columnwidth}{!}{%\n\\begin{tabular}")
        latex_full = latex_full.replace("\\end{tabular}", "\\end{tabular}\n}")
        latex_full += "\\end{table}\n"

        with open(f"tables/model_comparison_{metric}.tex", "w") as f:
            f.write(latex_full)


else:
    print("No data available for model comparison tables.")


In [ ]:
# (Deprecated) Single-metric table generation moved to the loop above.


In [ ]:
def plot_comparison_heatmap(df, percentage=1.0, metric="balanced_accuracy", figsize=(14, 6)):
    """Plot heatmap of model performance across tasks."""
    table = model_comparison_table(df, percentage, metric)
    
    if table is None or len(table) == 0:
        return
    
    # Remove Mean column for heatmap
    plot_table = table.drop(columns=["Mean"], errors="ignore")
    
    plt.figure(figsize=figsize)
    sns.heatmap(plot_table, annot=True, cmap="RdYlGn", fmt=".2f",
                vmin=0, vmax=1, linewidths=0.5,
                cbar_kws={"label": metric.replace("_", " ").title()})
    plt.title(f"Model {metric.replace('_', ' ').title()} by Task ({int(percentage*100)}% training data)",
              fontsize=14)
    plt.xlabel("Task", fontsize=12)
    plt.ylabel("Model", fontsize=12)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(f"results/comparison_heatmap_{metric}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to results/comparison_heatmap_{metric}.png")

In [ ]:
# Plot heatmap
if len(df) > 0:
    plot_comparison_heatmap(df, percentage=1.0, metric="balanced_accuracy")

## 4. Summary Statistics

In [ ]:
def best_model_per_task(df, percentage=1.0, metric="balanced_accuracy"):
    """Find the best performing model for each task."""
    subset = df[df["percentage"] == percentage].copy()
    
    if len(subset) == 0:
        return None
    
    # Group and find best
    max_acc = subset.groupby('task')['balanced_accuracy'].transform('max')
    best = subset[subset['balanced_accuracy'] == max_acc][['task', 'task_display', 'model_display', 'balanced_accuracy']].copy()
    best = best.rename(columns={metric: f"best_{metric}"})
    best = best.sort_values("task_display")
    
    return best

In [ ]:
# Best model per task
if len(df) > 0:
    best = best_model_per_task(df)
    if best is not None:
        print("Best Model per Task (100% training data)")
        display(best)

In [ ]:
#what model has the most bests? 
w = best['model_display'].value_counts()
w

In [ ]:
# def compute_model_rankings(df, percentage=1.0, metric="balanced_accuracy"):
#     """Compute average rank of each model across tasks."""
#     subset = df[df["percentage"] == percentage].copy()
    
#     if len(subset) == 0:
#         return None
    
#     # Compute rank per task (1 = best)
#     subset["rank"] = subset.groupby("task")[metric].rank(ascending=False)
    
#     # Average rank per model
#     rankings = subset.groupby("model_display").agg({
#         "rank": "mean",
#         metric: "mean"
#     }).round(2)
    
#     rankings = rankings.sort_values("rank")
#     rankings.columns = ["Avg Rank", f"Avg {metric.title()}"]
    
#     return rankings

def compute_model_rankings(df, percentage=1.0, metric="balanced_accuracy"):
    """Compute average rank of each model across tasks (only tasks with 2+ models)."""
    subset = df[df["percentage"] == percentage].copy()
    
    if len(subset) == 0:
        return None
    
    # Only include tasks that have multiple models
    task_model_counts = subset.groupby("task")["model_display"].nunique()
    tasks_with_multiple_models = task_model_counts[task_model_counts > 1].index
    print(len(tasks_with_multiple_models))
    subset = subset[subset["task"].isin(tasks_with_multiple_models)]
    
    if len(subset) == 0:
        print("No tasks with multiple models to rank")
        return None
    
    # Compute rank per task (1 = best)
    subset["rank"] = subset.groupby("task")[metric].rank(ascending=False)
    
    # Average rank per model
    rankings = subset.groupby("model_display").agg({
        "rank": "mean",
        metric: "mean"
    }).round(2)
    
    rankings = rankings.sort_values("rank")
    rankings.columns = ["Avg Rank", f"Avg {metric.title()}"]
    
    return rankings

In [ ]:
# Model rankings
if len(df) > 0:
    rankings = compute_model_rankings(df)
    if rankings is not None:
        print("Model Rankings (lower rank = better)")
        display(rankings)

## 5. BCI vs Clinical Comparison

In [ ]:
def plot_bci_vs_clinical(df, percentage=1.0, metric="accuracy"):
    """Compare model performance on BCI vs Clinical tasks."""
    subset = df[df["percentage"] == percentage].copy()
    
    if len(subset) == 0:
        return
    
    # Group by model and task type
    comparison = subset.groupby(["model_display", "task_type"])[metric].mean().unstack()
    comparison = comparison.sort_values("BCI", ascending=False) if "BCI" in comparison.columns else comparison
    
    # Plot
    ax = comparison.plot(kind="bar", figsize=(12, 6), width=0.8)
    plt.xlabel("Model", fontsize=12)
    plt.ylabel(metric.replace("_", " ").title(), fontsize=12)
    plt.title(f"BCI vs Clinical Performance ({int(percentage*100)}% training data)", fontsize=14)
    plt.xticks(rotation=45, ha="right")
    plt.legend(title="Task Type", fontsize=10)
    plt.ylim(0, 1)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("results/bci_vs_clinical.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved to results/bci_vs_clinical.png")

In [ ]:
# BCI vs Clinical comparison
if len(df) > 0:
    plot_bci_vs_clinical(df)

## 6. Export Results

In [ ]:
# Export full results to CSV
if len(df) > 0:
    output_path = "results/analysis_summary.csv"
    df.to_csv(output_path, index=False)
    print(f"Exported results to {output_path}")
    
    # Also export comparison table
    table = model_comparison_table(df, percentage=1.0)
    if table is not None:
        table.to_csv("results/model_comparison_table.csv")
        print("Exported comparison table to results/model_comparison_table.csv")

## 7. Custom Analysis

Use the cells below for your own analysis.

In [ ]:
# Example: Filter and analyze specific subset
# df_lejepa = df[df["model_display"] == "LeJEPA"]
# df_bci = df[df["task_type"] == "BCI"]
# df_low_data = df[df["percentage"] <= 0.1]

In [ ]:
# Example: Compare linear probe vs full fine-tuning (if both available)
# if "linear_probe" in df.columns and df["linear_probe"].nunique() > 1:
#     lp_comparison = df.groupby(["model_display", "linear_probe"])["accuracy"].mean().unstack()
#     display(lp_comparison)

## Additional Experiments: Noise Robustness

All plots above use clean-only results. This section visualizes the noise sweeps.


In [ ]:
df_for_noiseplot = df_combined[(df_combined["percentage"] == 1.0)]
clean_baseline = df_for_noiseplot[df_for_noiseplot["is_clean"]].groupby("model_display")["balanced_accuracy"].mean()

In [ ]:
noise_to_plot  = df_for_noiseplot[~df_for_noiseplot["is_clean"]].sort_values("noise_snr_db", ascending=False)
noise_to_plot['noise_types_str'] = noise_to_plot['noise_types_str'].fillna(noise_to_plot['noise_types'])
noise_to_plot['noise_types_str'] = noise_to_plot['noise_types_str'].replace('gaussian+one_over_f+emg+channel_dropout', 'all')
noise_to_plot['model_display'] = noise_to_plot['model_display'].str.split('_noise').apply(lambda x: x[0])

In [ ]:
noise_to_plot['task_display'] = noise_to_plot['task_display'].str.replace('_clinical', '').str.replace('_', ' ').str.capitalize()

In [ ]:
for task in noise_to_plot['task'].unique():
    plt.figure(figsize=(10, 6))
    task_data = noise_to_plot[noise_to_plot['task'] == task]
    task_data = task_data[task_data['noise_types_str'] == 'all']
    task_data = task_data[~task_data['model_display'].str.contains('20')]
    task_data = task_data[task_data['noise_snr_db'] >= 0]
    sns.lineplot(
        data=task_data,
        x="noise_snr_db",
        y="balanced_accuracy",
        hue="model_display",
        hue_order=list(model_colors.keys()),
        palette=model_colors,
        marker="o"
    )
    plt.gca().invert_xaxis()  # Higher noise to the right visually (optional)
    plt.title(f"{task_data['task_display'].iloc[0]}") #Noise Robustness on Task: 
    plt.xlabel("Noise SNR (dB)")
    plt.ylabel("Balanced Accuracy")
    plt.ylim(0, 1)
    plt.grid(True, alpha=0.3)
    plt.legend(title="Model", fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
noise_to_plot['task'].unique()

In [ ]:
#PAPER FIGURE!
sns.set_context('poster')
fig, axes = plt.subplots(2, 2, figsize=(12, 12), sharex=False, sharey=True)
for i, task in enumerate(['abnormal_clinical', 'epilepsy_clinical', 'parkinsons_clinical', 'seizure_clinical']):
    task_data = noise_to_plot[noise_to_plot['task'] == task]
    task_data = task_data[task_data['noise_types_str'] == 'all']
    task_data = task_data[~task_data['model_display'].str.contains('20')]
    task_data = task_data[task_data['noise_snr_db'] >= 0]
    ax = axes.flatten()[i]
    sns.lineplot(
        data=task_data,
        x="noise_snr_db",
        y="balanced_accuracy",
        hue="model_display",
        hue_order=list(model_colors.keys()),
        palette=model_colors,
        marker="o",
        ax=ax
    )
    ax.set_title(f"{task_data['task_display'].iloc[0]}") #Noise Robustness on Task: 
    ax.set_xlabel("Noise SNR (dB)")
    ax.set_ylabel("Balanced Accuracy")
    ax.set_ylim(.2, 1)
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(title="Model")
    else:
        ax.get_legend().remove()
    ax.invert_xaxis()

plt.tight_layout()
plt.savefig("figures/noise_robustness_clinical_tasks.png", dpi=300, bbox_inches="tight")

In [ ]:
df_all['probe_type'].unique()

In [ ]:

#Show results for attentive probe
attentive = df_all[df_all['probe_type'] == 'attentive']
attentive = attentive[attentive['is_clean']]

attentive.groupby(['model_display', 'task'])['balanced_accuracy'].mean().sort_values(ascending=False)

In [ ]:
import math
import seaborn as sns
import matplotlib.pyplot as plt

# Filter once
plot_df = noise_to_plot.copy()
plot_df = plot_df[~plot_df["model_display"].str.contains("20", na=False)]
plot_df = plot_df[plot_df["noise_snr_db"] >= 0]

for task, task_data in plot_df.groupby("task"):
    task_display = task_data["task_display"].iloc[0]
    noise_types = sorted(task_data["noise_types_str"].unique())
    #skip if not all models have not all noise types
    models_in_task = task_data["model_display"].unique()
    skip = False
    for model in models_in_task:
        model_data = task_data[task_data["model_display"] == model]
        model_noise_types = model_data["noise_types_str"].unique()
        if set(model_noise_types) != set(noise_types):
            skip = True
            break
    if skip:
        continue

    n = len(noise_types)
    cols = min(3, n)
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows), sharey=True, sharex=True)
    axes = axes.flatten() if n > 1 else [axes]

    for ax, nt in zip(axes, noise_types):
        sub = task_data[task_data["noise_types_str"] == nt]
        sns.lineplot(
            data=sub,
            x="noise_snr_db",
            y="balanced_accuracy",
            hue="model_display",
            hue_order=list(model_colors.keys()),
            palette=model_colors,
            marker="o",
            ax=ax
        )
        ax.invert_xaxis()
        ax.set_title(nt)
        ax.set_xlabel("Noise SNR (dB)")
        ax.set_ylabel("Balanced Accuracy")
        ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.3)
        ax.legend().remove()

    # Remove empty axes if any
    for ax in axes[len(noise_types):]:
        ax.remove()

    # Single shared legend
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")

    fig.suptitle(task_display, y=1.02)
    fig.tight_layout()
    plt.savefig(f"figures/noise/noise_robustness_{task}.png", dpi=300, bbox_inches="tight")
    plt.show()
